# Phase 2 — Automated Cleaning & Data-Quality Engine
### Enterprise Deposit Data Quality & Anomaly Detection Pipeline

This notebook implements **Phase 2** of the *Field Guide* (`Field Guide.docx`) exactly as
specified, and runs it against the **three real datasets collected for this project**:

| # | Dataset | Folder | Source |
|---|---------|--------|--------|
| 1 | HMDA Modified LAR 2025 | `HMDA_modified_LAR_2025/2025_combined_mlar.txt` | FFIEC / CFPB |
| 2 | CFPB Consumer Complaint Database | `CFPB_consumer_complaint_database/complaints.csv` | CFPB |
| 3 | SEC EDGAR Financial Statement Data Sets 2026Q1 | `SEC_EDGAR_financial_statement_2026q1/{sub,num,tag,pre}.txt` | SEC DERA |

**What Phase 2 does (per the Field Guide):**

1. **Data auditing & Health Score** — `audit_dataframe()` scans for null cells, exact-duplicate
   rows and structural type mismatches, returning a **0-100 Data Health Score** as a weighted
   penalty: **50 % nulls, 30 % duplicates, 20 % mixed types**.
2. **Type standardization** — `standardize_types()` strips `$`, commas and accounting parentheses
   from currency fields, coerces them to float, parses timestamps, normalises categorical casing.
3. **Segment-based imputation** — `impute_segment_median()` fills missing financial values with the
   **median of the row's own business segment/tier**, falling back to the global median.
4. **Safe deduplication** — `deduplicate()` drops duplicate rows on an explicit key (or full-row),
   keeping the first occurrence and reporting the count removed.

**Scale note.** The raw files are large (HMDA ~ 3.1 GB / ~13.5 M rows, CFPB ~ 8.6 GB,
SEC `num.txt` ~ 0.56 GB / ~3.85 M rows). Every stage below is therefore implemented as a
**two-pass, chunked** streaming engine so the *entire* file is processed without loading it whole
into memory. The core engine functions are identical to the Field Guide's; the chunked
orchestrator simply feeds them the file one block at a time.

## 0 - Setup, paths and logging
Set `SAMPLE_NROWS` to an integer for a fast trial run, or leave it as `None` to process the
**full** files (the default, as requested). `CHUNK_SIZE` controls the streaming block size.

In [1]:
from __future__ import annotations
import os, re, json, logging, hashlib
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
import numpy as np
import pandas as pd

# --- Project paths ----------------------------------------------------------
# This notebook lives in the project root alongside the three data folders.
PROJECT_DIR = os.getcwd()
DATA = {
    "hmda": os.path.join(PROJECT_DIR, "HMDA_modified_LAR_2025", "2025_combined_mlar.txt"),
    "cfpb": os.path.join(PROJECT_DIR, "CFPB_consumer_complaint_database", "complaints.csv"),
    "sec_num": os.path.join(PROJECT_DIR, "SEC_EDGAR_financial_statement_2026q1", "num.txt"),
    "sec_sub": os.path.join(PROJECT_DIR, "SEC_EDGAR_financial_statement_2026q1", "sub.txt"),
}
OUTDIR = os.path.join(PROJECT_DIR, "phase2_cleaned_output")
os.makedirs(OUTDIR, exist_ok=True)

# --- Run knobs --------------------------------------------------------------
SAMPLE_NROWS: Optional[int] = None   # None = process the FULL file. Set e.g. 200_000 for a quick test.
CHUNK_SIZE   = 250_000               # rows per streaming block
GLOBAL_DEDUP = True                  # drop duplicates across the whole file (uses a hash set)

# --- Logging: an audit trail, not print() -----------------------------------
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s | %(levelname)-7s | %(message)s",
                    datefmt="%H:%M:%S", force=True)
log = logging.getLogger("phase2")
log.info("Project dir: %s", PROJECT_DIR)
for k, v in DATA.items():
    log.info("  %-8s -> %s (%s)", k, v, "FOUND" if os.path.exists(v) else "MISSING")

15:59:35 | INFO    | Project dir: C:\Users\Trong Khoi Van\Desktop\Denison\Banking Deposit Data ML Project
15:59:35 | INFO    |   hmda     -> C:\Users\Trong Khoi Van\Desktop\Denison\Banking Deposit Data ML Project\HMDA_modified_LAR_2025\2025_combined_mlar.txt (FOUND)
15:59:35 | INFO    |   cfpb     -> C:\Users\Trong Khoi Van\Desktop\Denison\Banking Deposit Data ML Project\CFPB_consumer_complaint_database\complaints.csv (FOUND)
15:59:35 | INFO    |   sec_num  -> C:\Users\Trong Khoi Van\Desktop\Denison\Banking Deposit Data ML Project\SEC_EDGAR_financial_statement_2026q1\num.txt (FOUND)
15:59:35 | INFO    |   sec_sub  -> C:\Users\Trong Khoi Van\Desktop\Denison\Banking Deposit Data ML Project\SEC_EDGAR_financial_statement_2026q1\sub.txt (FOUND)


## 1 - `Config`: map each dataset's columns onto logical roles
The Field Guide's central design: columns are **not hard-coded** - each dataset maps its real
columns onto logical *roles* (amount, segment, id, timestamp, channel). The same engine then runs
on all three. Below is one `Config` per dataset.

In [2]:
@dataclass
class Config:
    """Central per-dataset configuration (logical role -> real column)."""
    name: str
    path: str
    # ---- file parsing --------------------------------------------------------
    sep: str = ","
    header: Optional[int] = 0
    names: Optional[List[str]] = None          # column names when file has no header
    usecols: Optional[List[str]] = None
    na_values: List[str] = field(default_factory=lambda: ["", "NA", "NaN", "Exempt", "exempt"])
    dtype: str = "object"                       # read everything as object, coerce later
    # ---- logical roles -------------------------------------------------------
    amount_cols: List[str] = field(default_factory=list)   # currency / numeric fields to clean+impute
    segment_col: Optional[str] = None                      # business tier for segment-median imputation
    account_id_col: Optional[str] = None
    timestamp_col: Optional[str] = None
    channel_col: Optional[str] = None
    categorical_cols: List[str] = field(default_factory=list)  # extra cols to case/whitespace-normalise
    # ---- behaviour -----------------------------------------------------------
    duplicate_subset: Optional[List[str]] = None  # None = full-row dedup
    timestamp_is_yyyymmdd: bool = False           # SEC ddate style integer YYYYMMDD


### HMDA Modified LAR - 85-field schema
The Modified LAR is **pipe-delimited with no header row**; the official 2025 field order
(FFIEC/CFPB Modified LAR Schema) is applied below. Currency/numeric fields contain regulatory
*"dirt"* - `Exempt` strings, `NA`/sentinel codes (e.g. `8888`, `9999`) and heavy right tails -
exactly the messiness the engine is built to repair.

In [3]:
# Official 2025 HMDA Modified LAR field order (data field 1..85), snake_cased.
HMDA_COLUMNS = [
    "activity_year","lei","loan_type","loan_purpose","preapproval","construction_method",
    "occupancy_type","loan_amount","action_taken","state","county","census_tract",
    "ethnicity_applicant_1","ethnicity_applicant_2","ethnicity_applicant_3","ethnicity_applicant_4","ethnicity_applicant_5",
    "ethnicity_co_applicant_1","ethnicity_co_applicant_2","ethnicity_co_applicant_3","ethnicity_co_applicant_4","ethnicity_co_applicant_5",
    "ethnicity_applicant_observed","ethnicity_co_applicant_observed",
    "race_applicant_1","race_applicant_2","race_applicant_3","race_applicant_4","race_applicant_5",
    "race_co_applicant_1","race_co_applicant_2","race_co_applicant_3","race_co_applicant_4","race_co_applicant_5",
    "race_applicant_observed","race_co_applicant_observed",
    "sex_applicant","sex_co_applicant","sex_applicant_observed","sex_co_applicant_observed",
    "age_applicant","age_applicant_ge_62","age_co_applicant","age_co_applicant_ge_62",
    "income","type_of_purchaser","rate_spread","hoepa_status","lien_status",
    "credit_scoring_model_applicant","credit_scoring_model_co_applicant",
    "denial_reason_1","denial_reason_2","denial_reason_3","denial_reason_4",
    "total_loan_costs","total_points_and_fees","origination_charges","discount_points","lender_credits",
    "interest_rate","prepayment_penalty_term","debt_to_income_ratio","combined_loan_to_value_ratio",
    "loan_term","introductory_rate_period","balloon_payment","interest_only_payments",
    "negative_amortization","other_nonamortizing_features","property_value",
    "manufactured_home_secured_property_type","manufactured_home_land_property_interest",
    "total_units","multifamily_affordable_units","submission_of_application","initially_payable_to_institution",
    "aus_1","aus_2","aus_3","aus_4","aus_5",
    "reverse_mortgage","open_end_line_of_credit","business_or_commercial_purpose",
]
assert len(HMDA_COLUMNS) == 85

CFG_HMDA = Config(
    name="hmda", path=DATA["hmda"], sep="|", header=None, names=HMDA_COLUMNS,
    # Read only the columns we actually use (faster + lighter), keeping demographics out.
    usecols=["activity_year","lei","loan_type","loan_purpose","occupancy_type","loan_amount",
             "action_taken","state","county","census_tract","income","loan_term","interest_rate",
             "rate_spread","total_loan_costs","origination_charges","discount_points","lender_credits",
             "property_value","lien_status","total_units"],
    amount_cols=["loan_amount","income","property_value","interest_rate","rate_spread",
                 "total_loan_costs","origination_charges","discount_points","lender_credits","loan_term"],
    segment_col="derived_loan_product_type",   # engineered below from loan_type + lien_status
    account_id_col="lei", timestamp_col="activity_year", channel_col="loan_purpose",
    categorical_cols=["state","action_taken","occupancy_type"],
    duplicate_subset=None,
    na_values=["","NA","NaN","Exempt","exempt","8888","9999","1111"],
)

### CFPB Consumer Complaint Database
This dataset is **text/categorical** with no currency fields, so `amount_cols` is empty. Phase 2
still audits it, parses its inconsistent date strings, and normalises categoricals; the heavy
missingness lives in *narrative*, *sub-issue*, *ZIP* and *tags*. (Numeric features for the anomaly
model are engineered in Phase 3.)

In [4]:
CFG_CFPB = Config(
    name="cfpb", path=DATA["cfpb"], sep=",", header=0,
    usecols=["Date received","Product","Sub-product","Issue","Sub-issue",
             "Consumer complaint narrative","Company","State","ZIP code","Tags",
             "Submitted via","Date sent to company","Company response to consumer",
             "Timely response?","Complaint ID"],
    amount_cols=[],                       # no currency columns in this dataset
    segment_col="Product",               # business segment = product line
    account_id_col="Complaint ID",
    timestamp_col="Date received",
    channel_col="Submitted via",
    categorical_cols=["Product","Sub-product","Company response to consumer","Timely response?"],
    duplicate_subset=["Complaint ID"],   # complaint id is the natural key
)

### SEC EDGAR Financial Statement Data Sets
The numeric facts live in `num.txt` (millions of rows); contextual metadata (form type, SIC,
fiscal period, registrant) lives in `sub.txt`. Per the Field Guide this requires a **multi-file
join on `adsh`**. `sub.txt` is small and is loaded once, in full, then broadcast onto each
streamed `num` chunk. The numeric fact `value` is the amount field; the filing `form` is the
business segment.

In [5]:
CFG_SEC = Config(
    name="sec", path=DATA["sec_num"], sep="\t", header=0,
    usecols=["adsh","tag","version","ddate","qtrs","uom","segments","coreg","value"],
    amount_cols=["value"],
    segment_col="form",                  # added by the adsh join with sub.txt
    account_id_col="cik",                # added by the join
    timestamp_col="ddate",               # YYYYMMDD integer
    channel_col="uom",
    categorical_cols=["form","uom","tag"],
    # A fact is uniquely keyed by adsh+tag+version+ddate+qtrs+uom+segments+coreg
    # (segments = dimensional members; excluding it would collapse legitimate breakdowns).
    duplicate_subset=["adsh","tag","version","ddate","qtrs","uom","segments","coreg"],
    timestamp_is_yyyymmdd=True,
)

# sub.txt loaded once (small) - the right side of the adsh join.
def load_sec_sub() -> pd.DataFrame:
    sub = pd.read_csv(DATA["sec_sub"], sep="\t", dtype="object",
                      usecols=["adsh","cik","name","sic","form","period","fy","fp"])
    log.info("[SEC] loaded sub.txt: %d filings", len(sub))
    return sub

## 2 - The cleaning engine (Field-Guide functions)
Each function is small, side-effect-free and testable. The chunked orchestrator in section 3 calls them.

In [6]:
# ---- PHASE 2.2 : type standardization -------------------------------------
def _clean_currency(series: pd.Series) -> pd.Series:
    """Strip $, commas, accounting parentheses -> float. '(1,234.50)' -> -1234.50."""
    s = series.astype(str).str.strip()
    neg = s.str.startswith("(") & s.str.endswith(")")
    s = s.str.replace(r"[(),$]", "", regex=True)
    out = pd.to_numeric(s, errors="coerce")
    out.loc[neg] = -out.loc[neg].abs()
    return out

def standardize_types(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    """Coerce currency->float, timestamp->datetime, categoricals->trimmed UPPER."""
    df = df.copy()
    for col in cfg.amount_cols:
        if col in df.columns:
            df[col] = _clean_currency(df[col])
    if cfg.timestamp_col and cfg.timestamp_col in df.columns:
        ts = df[cfg.timestamp_col]
        if cfg.timestamp_is_yyyymmdd:
            df["event_timestamp"] = pd.to_datetime(ts.astype(str), format="%Y%m%d", errors="coerce")
        else:
            # Cast to str first so a 4-digit year parses to Jan-1 of that year.
            df["event_timestamp"] = pd.to_datetime(ts.astype(str), errors="coerce")
    for col in set([c for c in [cfg.segment_col, cfg.channel_col] if c] + cfg.categorical_cols):
        if col in df.columns and df[col].dtype == object:
            df[col] = df[col].astype(str).str.strip().str.upper().replace({"NAN": np.nan})
    return df

# ---- PHASE 2.3 : segment-based imputation (apply precomputed medians) ------
def impute_segment_median(df: pd.DataFrame, cfg: Config,
                          seg_medians: Dict[str, Dict], global_medians: Dict[str, float]
                          ) -> Tuple[pd.DataFrame, Dict[str, int]]:
    """Fill missing amounts with the median of the row's own segment; fall back to global."""
    df = df.copy()
    filled = {}
    seg = cfg.segment_col
    for col in cfg.amount_cols:
        if col not in df.columns:
            continue
        n_missing = int(df[col].isna().sum())
        if n_missing == 0:
            continue
        if seg and seg in df.columns and col in seg_medians:
            df[col] = df[col].fillna(df[seg].map(seg_medians[col]))
        df[col] = df[col].fillna(global_medians.get(col, np.nan))
        filled[col] = n_missing
    return df, filled


In [7]:
# ---- PHASE 2.1 : streaming auditor + Data Health Score --------------------
class StreamingAuditor:
    """Accumulates null / duplicate / mixed-type stats across chunks, then scores 0-100."""
    def __init__(self, label: str):
        self.label = label
        self.rows = 0; self.cols = 0
        self.null_cells = 0
        self.nulls_by_col = {}
        self.dup_rows = 0
        self.mixed_type_cols = []
        self._mixed_done = False

    def _detect_mixed(self, df: pd.DataFrame):
        for col in df.select_dtypes(include="object").columns:
            sample = df[col].dropna().astype(str).head(5000)
            if sample.empty:
                continue
            frac_numeric = sample.str.match(r"^-?\$?[\d,]+(\.\d+)?$").mean()
            if 0.05 < frac_numeric < 0.95:
                self.mixed_type_cols.append(col)
        self._mixed_done = True

    def update(self, df: pd.DataFrame, dup_in_chunk: int = 0):
        if self.cols == 0:
            self.cols = df.shape[1]
        if not self._mixed_done:
            self._detect_mixed(df)
        self.rows += len(df)
        self.null_cells += int(df.isna().sum().sum())
        nb = df.isna().sum()
        for c, v in nb.items():
            self.nulls_by_col[c] = self.nulls_by_col.get(c, 0) + int(v)
        self.dup_rows += int(dup_in_chunk)

    def report(self) -> Dict:
        total_cells = max(self.rows * self.cols, 1)
        null_rate = self.null_cells / total_cells
        dup_rate = self.dup_rows / max(self.rows, 1)
        mixed_type_rate = len(self.mixed_type_cols) / max(self.cols, 1)
        penalty = 0.50*null_rate + 0.30*dup_rate + 0.20*mixed_type_rate
        health = round(max(0.0, 1.0 - penalty) * 100, 2)
        return {
            "label": self.label, "rows": self.rows, "cols": self.cols,
            "null_cells": self.null_cells, "null_rate": round(null_rate, 4),
            "duplicate_rows": self.dup_rows, "duplicate_rate": round(dup_rate, 4),
            "mixed_type_columns": self.mixed_type_cols, "mixed_type_rate": round(mixed_type_rate, 4),
            "health_score": health,
            "nulls_by_col": {k: v for k, v in sorted(self.nulls_by_col.items(),
                                                     key=lambda x: -x[1]) if v > 0},
        }


## 3 - Chunked two-pass orchestrator
Because medians and global deduplication need the *whole* file, the engine makes two streaming
passes:

* **Pass A** reads only the segment + amount columns to compute **exact per-segment medians**
  (and global medians as the fallback).
* **Pass B** streams the file again, applying `standardize_types -> impute_segment_median ->
  deduplicate`, auditing *before* and *after*, and writing the cleaned rows to disk chunk-by-chunk.

Global dedup keeps a set of row hashes so the first occurrence wins across chunk boundaries.

In [8]:
def _chunk_reader(cfg: Config, usecols=None):
    """Yield raw chunks for a dataset, respecting SAMPLE_NROWS."""
    kw = dict(sep=cfg.sep, header=cfg.header, dtype=cfg.dtype,
              na_values=cfg.na_values, keep_default_na=True,
              chunksize=CHUNK_SIZE, on_bad_lines="skip")
    if cfg.names is not None:
        kw["names"] = cfg.names
    if usecols is not None:
        kw["usecols"] = usecols
    elif cfg.usecols is not None:
        kw["usecols"] = cfg.usecols
    reader = pd.read_csv(cfg.path, **kw)
    seen = 0
    for chunk in reader:
        if SAMPLE_NROWS is not None and seen + len(chunk) > SAMPLE_NROWS:
            chunk = chunk.iloc[: SAMPLE_NROWS - seen]
        seen += len(chunk)
        yield chunk
        if SAMPLE_NROWS is not None and seen >= SAMPLE_NROWS:
            break


SEC_SUB_CACHE = None

def _prep_dataset(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    """Dataset-specific preparation BEFORE the generic engine: joins + engineered segments."""
    global SEC_SUB_CACHE
    df = df.copy()
    if cfg.name == "hmda":
        # Engineer the product-tier segment the Field Guide expects.
        lt = {"1":"CONVENTIONAL","2":"FHA","3":"VA","4":"RHS_FSA"}
        ls = {"1":"FIRST_LIEN","2":"SUBORDINATE"}
        seg = df["loan_type"].astype(str).map(lt).fillna("OTHER")
        if "lien_status" in df.columns:
            seg = seg + "|" + df["lien_status"].astype(str).map(ls).fillna("NA")
        df["derived_loan_product_type"] = seg
    elif cfg.name == "sec":
        if SEC_SUB_CACHE is None:
            SEC_SUB_CACHE = load_sec_sub()
        df = df.merge(SEC_SUB_CACHE, on="adsh", how="left")
    # Standardize types (currency/date/categoricals) - needed before median maths.
    df = standardize_types(df, cfg)
    return df


def _pass_a_medians(cfg: Config) -> Tuple[Dict[str, Dict], Dict[str, float]]:
    """Compute exact per-segment medians + global medians for the amount columns."""
    if not cfg.amount_cols:
        return {}, {}
    frames = []
    for chunk in _chunk_reader(cfg):                # full row needed to engineer/join segment
        chunk = _prep_dataset(chunk, cfg)
        keep = [c for c in cfg.amount_cols if c in chunk.columns]
        extra = [cfg.segment_col] if (cfg.segment_col and cfg.segment_col in chunk.columns) else []
        frames.append(chunk[keep + extra].copy())
    allv = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    global_medians = {c: float(allv[c].median()) for c in cfg.amount_cols if c in allv.columns}
    seg_medians = {}
    if cfg.segment_col and cfg.segment_col in allv.columns:
        for c in cfg.amount_cols:
            if c in allv.columns:
                seg_medians[c] = allv.groupby(cfg.segment_col)[c].median().to_dict()
    return seg_medians, global_medians


def _row_hashes(df: pd.DataFrame, subset: Optional[List[str]]) -> pd.Series:
    cols = subset if subset else list(df.columns)
    cols = [c for c in cols if c in df.columns]
    return pd.util.hash_pandas_object(df[cols].astype(str), index=False)


def _prep_before_audit(chunk: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    """A view of the RAW chunk (with SEC join) for the 'before' audit / health score."""
    global SEC_SUB_CACHE
    if cfg.name == "sec":
        if SEC_SUB_CACHE is None:
            SEC_SUB_CACHE = load_sec_sub()
        return chunk.merge(SEC_SUB_CACHE, on="adsh", how="left")
    if cfg.name == "hmda":
        c = chunk.copy(); c["derived_loan_product_type"] = np.nan
        return c
    return chunk


In [9]:
def clean_dataset(cfg: Config) -> Dict:
    """Full chunked Phase-2 run for one dataset. Returns the before/after health summary."""
    log.info("=" * 78)
    log.info("CLEANING DATASET: %s", cfg.name.upper())
    log.info("=" * 78)

    # ---- Pass A: medians ----------------------------------------------------
    log.info("[%s] Pass A - computing segment & global medians ...", cfg.name)
    seg_medians, global_medians = _pass_a_medians(cfg)
    log.info("[%s] global medians: %s", cfg.name,
             {k: round(v, 2) for k, v in global_medians.items()})

    # ---- Pass B: standardize -> impute -> dedup -> write --------------------
    out_path = os.path.join(OUTDIR, f"{cfg.name}_cleaned.csv")
    before = StreamingAuditor("before"); after = StreamingAuditor("after")
    seen_hashes = set() if GLOBAL_DEDUP else None
    total_dupes = 0; total_filled = {}; wrote_header = False; n_out = 0

    for chunk in _chunk_reader(cfg):
        before.update(_prep_before_audit(chunk, cfg))   # raw view for BEFORE audit
        df = _prep_dataset(chunk, cfg)                  # joins + standardize
        df, filled = impute_segment_median(df, cfg, seg_medians, global_medians)
        for k, v in filled.items():
            total_filled[k] = total_filled.get(k, 0) + v
        # ---- safe deduplication (global, first occurrence wins) -------------
        h = _row_hashes(df, cfg.duplicate_subset)
        dup_mask = h.duplicated(keep="first")           # within-chunk dupes
        if seen_hashes is not None:
            dup_mask = dup_mask | h.isin(seen_hashes)   # cross-chunk dupes
        n_dup = int(dup_mask.sum()); total_dupes += n_dup
        df = df.loc[~dup_mask].reset_index(drop=True)
        if seen_hashes is not None:
            seen_hashes.update(h.loc[~dup_mask].tolist())
        after.update(df)
        df.to_csv(out_path, mode=("w" if not wrote_header else "a"),
                  index=False, header=not wrote_header)
        wrote_header = True; n_out += len(df)

    rep_before = before.report(); rep_after = after.report()
    rep_before["duplicate_rows"] = total_dupes
    rep_before["duplicate_rate"] = round(total_dupes / max(rep_before["rows"], 1), 4)
    p = (0.50*rep_before["null_rate"] + 0.30*rep_before["duplicate_rate"]
         + 0.20*rep_before["mixed_type_rate"])
    rep_before["health_score"] = round(max(0.0, 1.0 - p) * 100, 2)

    summary = {
        "dataset": cfg.name, "rows_in": rep_before["rows"], "rows_out": n_out,
        "duplicates_removed": total_dupes, "values_imputed": total_filled,
        "before": rep_before, "after": rep_after,
        "health_improvement": round(rep_after["health_score"] - rep_before["health_score"], 2),
        "output_csv": out_path,
    }
    log.info("[%s] DONE  health %.1f -> %.1f | dupes removed %d | imputed %s | rows %d -> %d",
             cfg.name, rep_before["health_score"], rep_after["health_score"],
             total_dupes, total_filled, rep_before["rows"], n_out)
    pd.DataFrame([
        {"stage":"before","rows":rep_before["rows"],"null_rate":rep_before["null_rate"],
         "duplicate_rows":rep_before["duplicate_rows"],"health_score":rep_before["health_score"]},
        {"stage":"after","rows":rep_after["rows"],"null_rate":rep_after["null_rate"],
         "duplicate_rows":0,"health_score":rep_after["health_score"]},
    ]).to_csv(os.path.join(OUTDIR, f"{cfg.name}_data_health_summary.csv"), index=False)
    return summary


## 4 - Run Phase 2 on all three datasets
Each call streams the **entire** file (set `SAMPLE_NROWS` above to cap for a quick test) and writes
`<dataset>_cleaned.csv` plus `<dataset>_data_health_summary.csv` into `phase2_cleaned_output/`.
Those cleaned CSVs are the inputs to the **Phase 3** notebook.

In [10]:
summaries = {}
for cfg in (CFG_HMDA, CFG_CFPB, CFG_SEC):
    summaries[cfg.name] = clean_dataset(cfg)

with open(os.path.join(OUTDIR, "phase2_run_summary.json"), "w") as f:
    json.dump(summaries, f, indent=2, default=str)
print("Phase 2 complete. Outputs in:", OUTDIR)

15:59:36 | INFO    | ==============================================================================
15:59:36 | INFO    | CLEANING DATASET: HMDA
15:59:36 | INFO    | ==============================================================================
15:59:36 | INFO    | [hmda] Pass A - computing segment & global medians ...
16:06:07 | INFO    | [hmda] global medians: {'loan_amount': 235000.0, 'income': 108.0, 'property_value': 395000.0, 'interest_rate': 6.62, 'rate_spread': 0.27, 'total_loan_costs': 5987.25, 'origination_charges': 1795.0, 'discount_points': 2664.41, 'lender_credits': 500.0, 'loan_term': 360.0}
16:25:49 | INFO    | [hmda] DONE  health 88.3 -> 99.9 | dupes removed 74764 | imputed {'income': 2190103, 'property_value': 3107132, 'interest_rate': 4854441, 'rate_spread': 7136925, 'total_loan_costs': 7404508, 'origination_charges': 7337898, 'discount_points': 10457361, 'lender_credits': 11514980, 'loan_term': 474247} | rows 13543542 -> 13468778
16:25:51 | INFO    | =================

Phase 2 complete. Outputs in: C:\Users\Trong Khoi Van\Desktop\Denison\Banking Deposit Data ML Project\phase2_cleaned_output


In [11]:
# ---- Before / After Data Health scorecard ---------------------------------
rows = []
for name, s in summaries.items():
    rows.append({
        "dataset": name.upper(),
        "rows_in": s["rows_in"], "rows_out": s["rows_out"],
        "health_before": s["before"]["health_score"],
        "health_after":  s["after"]["health_score"],
        "delta_health": s["health_improvement"],
        "null_rate_before": s["before"]["null_rate"],
        "null_rate_after":  s["after"]["null_rate"],
        "duplicates_removed": s["duplicates_removed"],
        "values_imputed": sum(s["values_imputed"].values()) if s["values_imputed"] else 0,
    })
scorecard = pd.DataFrame(rows).set_index("dataset")
scorecard

,rows_in,rows_out,health_before,health_after,delta_health,null_rate_before,null_rate_after,duplicates_removed,values_imputed
dataset,,,,,,,,,
HMDA,13543542,13468778,88.28,99.87,11.59,0.2311,0.0025,74764,54477595
CFPB,15694521,15694521,92.71,93.12,0.41,0.1191,0.1125,0,0
SEC,3690955,3690937,94.56,94.99,0.43,0.1089,0.1003,18,139723


## 5 - What was produced
For each of the three datasets, Phase 2 wrote:

* `phase2_cleaned_output/<dataset>_cleaned.csv` - every record, standardized + imputed + de-duplicated, with an `event_timestamp` column.
* `phase2_cleaned_output/<dataset>_data_health_summary.csv` - the 2-row before/after table that drives the Tableau **Data Health** tab.
* `phase2_cleaned_output/phase2_run_summary.json` - full audit metrics for all three runs.

Continue to **`phase3_anomaly_detection.ipynb`**, which reads these cleaned CSVs and runs the
unsupervised Isolation-Forest anomaly detector.